
## Patterns

### Pattern 1: Use Delta for Infinite Retention

TBLPROPERTIES (
  pipelines.reset.allowed=false
)

## Pattern 2 : Multiplex Ingestion



In [0]:


@dlt.table(
    table_properties={
        "pipelines.reset.allowed": "false"
    }
)
def date_lookup():
    return spark.read.table(f"{lookup_db}.date_lookup_clone").select("date", "week_part")

@dlt.table(
    partition_cols=["topic", "week_part"],
    table_properties={
        "quality": "bronze",
        "pipelines.reset.allowed": "false"
    }
)
def bronze():
    PATH_INPUT = "test"
    kafka_schema = "key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG"
    return(
        spark.readStream
            .format("json")
            .schema(kafka_schema)
            .option("cloudFiles.format", "json")
            .load(f"{PATH_INPUT}/daily")
            .join(
                F.broadcast(dlt.read("date_lookup")),
                F.to_date((F.col("timestamp") / 1000).cast("timestamp")) == F.col("date"), "left")
            )
    

    

In [0]:
@dlt.table
def distinct_topics():
    return dlt.read("bronze").select("topic").distinct()

## Create a Pipeline by code

## Run a Pipeline by code

In [0]:
demo_pipeline = DeclarativePipelineCreator(
    pipeline_name="demo_pipeline",
    catalog_name="prd",
    schema_name="l_bronze",
    root_path_folder_name = "Pipeline",
    source_folder_names=[
        'SDLT 2.1.1 - Auto Load to bronze',
        'SDLT 2.2.1 - Bronze to Silver',
        'SDLT 2.3.1 - Data Quality Enforcement'
    ],
    configuration={"source": "/Volumes/prd/demo_volumes/rand_engine_data/dlt/" , "lookup_db": 'db_lookup'},
    serverless=True
)

demo_pipeline.create_pipeline()

## Flagging

In [0]:
rules = {
    "valid_heartrate": "heartrate IS NOT NULL",
    "valid_device_id": "device_id IS NOT NULL",
    "valid_Device_id_range"
}

@dlt.table(
    table_properties={"quality": "silver"}
)
@dlt_expect_all_or_drop(rules)
def bpm_silver():
    return(
        dlt.read_stream("bmp_bronze")
        .select("*", F.when(F.col("heartrate") <= 0 "Negative BPM").otherwise("OK").alias("bpm_check"))
        .withWatermark("time", "30 seconds")
        .dropDuplicates("device_id", "time")
    )

# Quarantine table



In [0]:
quarantine_rules = {}
quarantine_riles["invalid_record"] = f"NOT ({' AND '.join(rules.values())})"
@dlt.table
@dlt.expect_all_or_drop(quarantine_rules)
def bpm_quarantine():
    return (
        dlt.read_stream("bpm_bronze")
    )